# 🎬 Scalable Video Recommendation System
## Step 1: 数据 Pipeline — KuaiRec 版本

---
**数据集**: KuaiRec（快手真实短视频推荐数据）

| 文件 | 说明 |
|------|------|
| `small_matrix.csv` | 1411用户×3327视频 全量交互矩阵（密度99.6%）|
| `big_matrix.csv` | 7176用户×10728视频 大规模交互矩阵 |
| `user_features.csv` | 用户侧特征（人口属性 + 设备信息）|
| `item_daily_features.csv` | 视频每日统计特征 |
| `item_categories.csv` | 视频类别标签 |

**本 Notebook 完成**:
```
Cell 1  → 安装依赖
Cell 2  → 配置 & 下载 KuaiRec 数据
Cell 3  → 数据探索 (EDA)
Cell 4  → 数据清洗
Cell 5  → 用户特征工程
Cell 6  → 视频特征工程
Cell 7  → 序列特征工程
Cell 8  → 时序样本划分 + 负采样
Cell 9  → Feature Store 写入
Cell 10 → PSI 数据质量监控
Cell 11 → Step 1 完成报告
```

> 💡 **建议开启 GPU**: 运行时 → 更改运行时类型 → T4 GPU

---
## Cell 1 — 安装依赖

In [ ]:
!pip install -q pyarrow pandas numpy scikit-learn matplotlib seaborn tqdm tabulate
print('✅ 依赖安装完成')

---
## Cell 2 — 配置 & 下载 KuaiRec 数据

In [ ]:
import os, time, logging, warnings, pickle
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

# ── 路径配置 ──
USE_GDRIVE = False  # 用了 Google Drive 改为 True
if USE_GDRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/VideoRecSys'
else:
    BASE_DIR = '/content/VideoRecSys'

RAW_DIR = f'{BASE_DIR}/data/raw/kuairec'
PATHS = {
    'raw'          : RAW_DIR,
    'cleaned'      : f'{BASE_DIR}/data/cleaned',
    'features'     : f'{BASE_DIR}/data/features',
    'samples'      : f'{BASE_DIR}/data/samples',
    'feature_store': f'{BASE_DIR}/feature_store',
    'logs'         : f'{BASE_DIR}/logs',
    'reports'      : f'{BASE_DIR}/reports',
}
for p in PATHS.values(): os.makedirs(p, exist_ok=True)

# ── 超参数 ──
CFG = {
    'use_small_matrix'     : True,   # True=small(1411用户) False=big(7176用户)
    'watch_ratio_threshold': 0.5,    # 正样本阈值: watch_ratio >= 0.5 为正样本
    'min_user_interactions': 10,     # 冷启动过滤: 用户最少交互数
    'min_item_interactions': 5,      # 冷启动过滤: 视频最少被交互数
    'neg_ratio'            : 4,      # 正负样本比 1:4
    'test_ratio'           : 0.2,    # 测试集比例
    'seq_len'              : 50,     # 序列最大长度
    'random_seed'          : 42,
}
np.random.seed(CFG['random_seed'])

# ── Logger ──
def get_logger():
    logger = logging.getLogger('Step1')
    logger.setLevel(logging.INFO)
    if logger.handlers: logger.handlers.clear()
    ch = logging.StreamHandler()
    ch.setFormatter(logging.Formatter('[%(asctime)s] %(message)s', '%H:%M:%S'))
    logger.addHandler(ch)
    return logger
log = get_logger()

class Timer:
    def __init__(self, name): self.name = name
    def __enter__(self): self.t = time.time(); log.info(f'▶ {self.name}'); return self
    def __exit__(self, *a): log.info(f'✅ {self.name} 完成  {time.time()-self.t:.1f}s')

# ── 下载 KuaiRec 数据 ──
zip_path = f'{RAW_DIR}/KuaiRec.zip'
if not os.path.exists(f'{RAW_DIR}/small_matrix.csv'):
    log.info('下载 KuaiRec 数据集...')
    !wget -q 'https://nas.chongminggao.top:4430/datasets/KuaiRec.zip' \
        --no-check-certificate \
        -O {zip_path}
    log.info('解压...')
    !unzip -q {zip_path} -d {RAW_DIR}
    # 将子目录文件移到 RAW_DIR
    !find {RAW_DIR}/KuaiRec -name '*.csv' -exec mv {{}} {RAW_DIR}/ \;
    !rm -rf {RAW_DIR}/KuaiRec {zip_path}
    log.info('KuaiRec 数据下载完成')
else:
    log.info('KuaiRec 数据已存在，跳过下载')

# 列出文件
for f in sorted(os.listdir(RAW_DIR)):
    size = os.path.getsize(f'{RAW_DIR}/{f}') / 1024**2
    print(f'  {f:40s}  {size:.1f} MB')

---
## Cell 3 — 数据探索 (EDA)

In [ ]:
with Timer('加载原始数据'):
    matrix_file = 'small_matrix.csv' if CFG['use_small_matrix'] else 'big_matrix.csv'
    inter_df    = pd.read_csv(f"{PATHS['raw']}/{matrix_file}")
    user_feat   = pd.read_csv(f"{PATHS['raw']}/user_features.csv")
    item_daily  = pd.read_csv(f"{PATHS['raw']}/item_daily_features.csv")
    item_cat    = pd.read_csv(f"{PATHS['raw']}/item_categories.csv")

print('\n【交互数据列名】')
print(inter_df.columns.tolist())
print(inter_df.head(3).to_string())

print('\n【数据规模】')
print(f'  交互数据:    {len(inter_df):,} 行  {inter_df.shape[1]} 列')
print(f'  用户特征:    {user_feat.shape}')
print(f'  视频每日特征: {item_daily.shape}')
print(f'  视频类别:    {item_cat.shape}')

print('\n【watch_ratio 分布】')
print(inter_df['watch_ratio'].describe())

print(f'\n  用户数: {inter_df["user_id"].nunique():,}')
print(f'  视频数: {inter_df["video_id"].nunique():,}')
print(f'  正样本率(watch_ratio>=0.5): {(inter_df["watch_ratio"]>=0.5).mean():.3f}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
plt.style.use('dark_background')

def styled(ax, title):
    ax.set_facecolor('#161b22')
    ax.set_title(title, color='#c9d1d9', fontsize=10)
    ax.tick_params(colors='#6e7681')
    for s in ax.spines.values(): s.set_color('#30363d')

fig = plt.figure(figsize=(14, 8))
fig.patch.set_facecolor('#0d1117')
gs  = gridspec.GridSpec(2, 3, figure=fig)

# watch_ratio 分布
ax1 = fig.add_subplot(gs[0, 0])
styled(ax1, 'watch_ratio 分布')
ax1.hist(inter_df['watch_ratio'].clip(0, 3), bins=60, color='#00d4ff', alpha=0.8)
ax1.axvline(x=CFG['watch_ratio_threshold'], color='#ff6b6b', lw=2,
            label=f'正样本阈值={CFG["watch_ratio_threshold"]}')
ax1.set_xlabel('watch_ratio', color='#6e7681')
ax1.legend(fontsize=8)

# 用户交互次数分布
ax2 = fig.add_subplot(gs[0, 1])
styled(ax2, '用户交互次数分布')
user_counts = inter_df['user_id'].value_counts()
ax2.hist(user_counts, bins=50, color='#39d353', alpha=0.8)
ax2.set_xlabel('交互次数', color='#6e7681')

# 视频被观看次数分布
ax3 = fig.add_subplot(gs[0, 2])
styled(ax3, '视频被观看次数分布')
item_counts = inter_df['video_id'].value_counts()
ax3.hist(item_counts, bins=50, color='#a371f7', alpha=0.8)
ax3.set_xlabel('被观看次数', color='#6e7681')

# 用户特征列示意
ax4 = fig.add_subplot(gs[1, :])
styled(ax4, '用户特征缺失率')
missing = user_feat.isnull().mean().sort_values(ascending=False).head(20)
ax4.bar(range(len(missing)), missing.values, color='#f0c040', alpha=0.8)
ax4.set_xticks(range(len(missing)))
ax4.set_xticklabels(missing.index, rotation=45, ha='right', fontsize=7)
ax4.set_ylabel('缺失率', color='#6e7681')

plt.tight_layout()
plt.savefig(f"{PATHS['reports']}/eda_kuairec.png",
            dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

---
## Cell 4 — 数据清洗

In [ ]:
with Timer('数据清洗'):
    df = inter_df.copy()

    # 统一列名: video_id → item_id
    df = df.rename(columns={'video_id': 'item_id'})

    # 处理 watch_ratio
    # KuaiRec 中 watch_ratio > 1 表示重复观看，是强正样本，保留但 clip 到 5
    df['watch_ratio'] = df['watch_ratio'].fillna(0).clip(0, 5)

    # 添加二值标签
    df['label'] = (df['watch_ratio'] >= CFG['watch_ratio_threshold']).astype(int)

    # 去重（同一用户对同一视频保留 watch_ratio 最大的一条）
    before = len(df)
    df = df.sort_values('watch_ratio', ascending=False)
    df = df.drop_duplicates(subset=['user_id', 'item_id'], keep='first')
    log.info(f'去重: {before:,} → {len(df):,}')

    # 处理时间戳
    # KuaiRec 的 timestamp 列名可能是 time 或 timestamp
    ts_col = 'time' if 'time' in df.columns else 'timestamp'
    if ts_col in df.columns:
        df['timestamp'] = pd.to_datetime(df[ts_col], unit='s', errors='coerce')
        df['timestamp_unix'] = df[ts_col]
    else:
        # KuaiRec small_matrix 没有时间戳，用 watch_ratio 排序生成伪时序
        log.info('无时间戳列，使用行索引生成伪时序')
        df = df.reset_index(drop=True)
        df['timestamp_unix'] = df.index

    # 冷启动过滤
    user_cnt = df['user_id'].value_counts()
    item_cnt = df['item_id'].value_counts()
    df = df[
        df['user_id'].isin(user_cnt[user_cnt >= CFG['min_user_interactions']].index) &
        df['item_id'].isin(item_cnt[item_cnt >= CFG['min_item_interactions']].index)
    ]
    log.info(f'冷启动过滤后: {len(df):,} 行')

    # 时序排序
    df = df.sort_values(['user_id', 'timestamp_unix']).reset_index(drop=True)

    # 保存
    df.to_parquet(f"{PATHS['cleaned']}/interactions_cleaned.parquet",
                  index=False, compression='snappy')

print(f'\n清洗完成:')
print(f'  行数:    {len(df):,}')
print(f'  用户数:  {df["user_id"].nunique():,}')
print(f'  视频数:  {df["item_id"].nunique():,}')
print(f'  正样本率: {df["label"].mean():.3f}')

---
## Cell 5 — 用户特征工程

In [ ]:
with Timer('用户特征工程'):
    # ── 行为统计特征（来自交互数据）──
    user_stats = df.groupby('user_id').agg(
        interaction_count =('item_id',       'count'),
        avg_watch_ratio   =('watch_ratio',   'mean'),
        std_watch_ratio   =('watch_ratio',   'std'),
        positive_rate     =('label',         'mean'),
        max_watch_ratio   =('watch_ratio',   'max'),
    ).reset_index()
    user_stats['std_watch_ratio'] = user_stats['std_watch_ratio'].fillna(0)

    # ── 活跃度分层（5级）──
    user_stats['activity_tier'] = pd.cut(
        user_stats['interaction_count'],
        bins=[0, 10, 50, 200, 500, float('inf')],
        labels=[0, 1, 2, 3, 4]
    ).astype(int)

    # ── 合并 KuaiRec 用户侧特征 ──
    uf = user_feat.copy()
    # 统一 ID 列名
    if 'user_id' not in uf.columns:
        uf = uf.rename(columns={uf.columns[0]: 'user_id'})

    # 只保留数值型特征，丢弃全空列
    num_cols = [c for c in uf.columns
                if c != 'user_id' and uf[c].dtype in
                [np.float64, np.float32, np.int64, np.int32]]
    uf = uf[['user_id'] + num_cols].fillna(0)

    # 合并
    final_user_feat = user_stats.merge(uf, on='user_id', how='left')
    final_user_feat = final_user_feat.fillna(0)

    # 只保留在清洗后数据中存在的用户
    valid_users = df['user_id'].unique()
    final_user_feat = final_user_feat[final_user_feat['user_id'].isin(valid_users)]

    final_user_feat.to_parquet(f"{PATHS['features']}/user_features.parquet",
                               index=False, compression='snappy')

print(f'用户特征: {final_user_feat.shape[0]:,} 用户  {final_user_feat.shape[1]} 列')
print(final_user_feat.head(2).to_string())

---
## Cell 6 — 视频特征工程

In [ ]:
with Timer('视频特征工程'):
    # ── 行为统计特征 ──
    item_stats = df.groupby('item_id').agg(
        play_count      =('user_id',      'count'),
        avg_watch_ratio =('watch_ratio',  'mean'),
        ctr_proxy       =('label',        'mean'),
        std_watch_ratio =('watch_ratio',  'std'),
    ).reset_index()
    item_stats['std_watch_ratio'] = item_stats['std_watch_ratio'].fillna(0)

    # ── 热度分层 ──
    item_stats['popularity_tier'] = pd.qcut(
        item_stats['play_count'], q=5, labels=[0,1,2,3,4], duplicates='drop'
    ).astype(int)

    # ── 合并视频类别（多热编码）──
    ic = item_cat.copy()
    id_col = 'video_id' if 'video_id' in ic.columns else ic.columns[0]
    ic = ic.rename(columns={id_col: 'item_id'})

    # 找到类别列（tag 或 category 相关）
    cat_cols = [c for c in ic.columns if 'tag' in c.lower() or 'cat' in c.lower()]
    if cat_cols:
        # 多热编码第一个类别列
        cat_col = cat_cols[0]
        dummies = ic.set_index('item_id')[cat_col].str.get_dummies(sep=',')
        dummies.columns = [f'cat_{c}' for c in dummies.columns[:20]]  # 只取前20个类别
        dummies = dummies.iloc[:, :20].reset_index()
        item_stats = item_stats.merge(dummies, on='item_id', how='left')

    # ── 合并视频每日特征（取均值聚合）──
    id_col2 = 'video_id' if 'video_id' in item_daily.columns else item_daily.columns[0]
    item_daily_renamed = item_daily.rename(columns={id_col2: 'item_id'})
    num_cols = [c for c in item_daily_renamed.columns
                if c != 'item_id' and item_daily_renamed[c].dtype in
                [np.float64, np.float32, np.int64, np.int32]]
    item_daily_agg = item_daily_renamed.groupby('item_id')[num_cols].mean().reset_index()
    item_stats = item_stats.merge(item_daily_agg, on='item_id', how='left')

    # 填充缺失值，只保留有效视频
    item_stats = item_stats.fillna(0)
    valid_items = df['item_id'].unique()
    item_stats = item_stats[item_stats['item_id'].isin(valid_items)]

    item_stats.to_parquet(f"{PATHS['features']}/item_features.parquet",
                          index=False, compression='snappy')

print(f'视频特征: {item_stats.shape[0]:,} 视频  {item_stats.shape[1]} 列')
print(item_stats.head(2).to_string())

---
## Cell 7 — 序列特征工程

In [ ]:
from tqdm import tqdm

with Timer('序列特征工程'):
    records = []
    for uid, group in tqdm(df.groupby('user_id'), desc='序列特征'):
        group = group.sort_values('timestamp_unix')
        all_seq = group['item_id'].tolist()
        pos_seq = group[group['label'] == 1]['item_id'].tolist()

        records.append({
            'user_id': uid,
            'all_seq': ','.join(map(str, all_seq[-CFG['seq_len']:])),
            'pos_seq': ','.join(map(str, pos_seq[-CFG['seq_len']:])),
            'seq_len': min(len(all_seq), CFG['seq_len']),
            'pos_seq_len': min(len(pos_seq), CFG['seq_len']),
        })

    seq_df = pd.DataFrame(records)
    seq_df.to_parquet(f"{PATHS['features']}/user_sequences.parquet",
                      index=False, compression='snappy')

print(f'序列特征: {len(seq_df):,} 用户')
print(f'平均序列长度: {seq_df["seq_len"].mean():.1f}')
print(f'平均正序列长度: {seq_df["pos_seq_len"].mean():.1f}')

---
## Cell 8 — 时序样本划分 + 负采样

In [ ]:
# ── 时序划分 ──
with Timer('时序划分'):
    train_list, test_list = [], []
    for uid, group in df.groupby('user_id'):
        group = group.sort_values('timestamp_unix')
        n_test = max(1, int(len(group) * CFG['test_ratio']))
        train_list.append(group.iloc[:-n_test])
        test_list.append(group.iloc[-n_test:])

    train_pos = pd.concat(train_list).reset_index(drop=True)
    test_pos  = pd.concat(test_list).reset_index(drop=True)
    log.info(f'训练正样本: {len(train_pos):,}  测试正样本: {len(test_pos):,}')

# ── 负采样（只对训练集负采样，测试集保留全部正负）──
with Timer('负采样'):
    all_items = list(df['item_id'].unique())
    item_popularity = df['item_id'].value_counts()
    hot_pool = item_popularity.nlargest(2000).index.tolist()

    neg_records = []
    for uid, group in tqdm(train_pos.groupby('user_id'), desc='负采样'):
        interacted = set(group['item_id'].tolist())
        n_neg = len(group) * CFG['neg_ratio']

        # 50% 随机 + 50% 热门
        cands   = [i for i in all_items if i not in interacted]
        n_rand  = min(int(n_neg * 0.5), len(cands))
        rand_negs = np.random.choice(cands, size=n_rand, replace=False)

        hot_cands = [i for i in hot_pool if i not in interacted]
        n_hot   = min(int(n_neg * 0.5), len(hot_cands))
        hot_negs  = np.random.choice(hot_cands, size=n_hot, replace=False)

        for iid in list(rand_negs) + list(hot_negs):
            neg_records.append({
                'user_id'       : uid,
                'item_id'       : iid,
                'watch_ratio'   : 0.0,
                'label'         : 0,
                'timestamp_unix': group['timestamp_unix'].max(),
            })

    train_neg = pd.DataFrame(neg_records)
    train_df  = pd.concat([train_pos, train_neg]).sample(
        frac=1, random_state=CFG['random_seed']).reset_index(drop=True)

    # 测试集：原始数据中包含正负（watch_ratio < 阈值为负样本）
    test_df = test_pos.copy()

    # 保存
    train_df.to_parquet(f"{PATHS['samples']}/train.parquet", index=False, compression='snappy')
    test_df.to_parquet(f"{PATHS['samples']}/test.parquet",   index=False, compression='snappy')

print(f'\n训练集: {len(train_df):,}  正样本率: {train_df["label"].mean():.3f}')
print(f'测试集: {len(test_df):,}   正样本率: {test_df["label"].mean():.3f}')

---
## Cell 9 — Feature Store 写入

In [ ]:
with Timer('Feature Store 写入'):
    # 加载特征
    uf_final = pd.read_parquet(f"{PATHS['features']}/user_features.parquet")
    if_final = pd.read_parquet(f"{PATHS['features']}/item_features.parquet")
    sq_final = pd.read_parquet(f"{PATHS['features']}/user_sequences.parquet")

    # 构建内存缓存（模拟 Redis）
    feature_store = {}

    for _, row in uf_final.iterrows():
        feature_store[f"user:{row['user_id']}"] = row.to_dict()

    for _, row in if_final.iterrows():
        feature_store[f"item:{row['item_id']}"] = row.to_dict()

    for _, row in sq_final.iterrows():
        feature_store[f"seq:{row['user_id']}"] = {
            'all_seq': row['all_seq'],
            'pos_seq': row['pos_seq'],
            'seq_len': row['seq_len'],
        }

    # 保存快照
    snap_path = f"{PATHS['feature_store']}/snapshot.pkl"
    with open(snap_path, 'wb') as f:
        pickle.dump(feature_store, f)

snap_mb = os.path.getsize(snap_path) / 1024**2
print(f'Feature Store: {len(feature_store):,} keys  {snap_mb:.2f} MB')

---
## Cell 10 — PSI 数据质量监控

In [ ]:
def compute_psi(base, curr, bins=10):
    base = np.array(base.dropna())
    curr = np.array(curr.dropna())
    breakpoints = np.unique(np.percentile(base, np.linspace(0, 100, bins+1)))
    if len(breakpoints) < 2: return 0.0
    bp = np.histogram(base, bins=breakpoints)[0]
    cp = np.histogram(curr, bins=breakpoints)[0]
    bp = np.where(bp==0, 1e-6, bp/bp.sum())
    cp = np.where(cp==0, 1e-6, cp/cp.sum())
    return float(np.sum((cp-bp)*np.log(cp/bp)))

# 训练集 vs 测试集特征分布对比
train_uf = uf_final[uf_final['user_id'].isin(train_df['user_id'].unique())]
test_uf  = uf_final[uf_final['user_id'].isin(test_df['user_id'].unique())]

monitor_feats = ['interaction_count', 'avg_watch_ratio', 'positive_rate',
                 'activity_tier', 'std_watch_ratio']
monitor_feats = [f for f in monitor_feats if f in uf_final.columns]

from tabulate import tabulate
rows = []
for feat in monitor_feats:
    psi = compute_psi(train_uf[feat], test_uf[feat])
    if psi < 0.1:   status = '✅ 稳定'
    elif psi < 0.2: status = '⚠️  轻微漂移'
    else:           status = '🚨 显著漂移'
    rows.append([feat, f'{psi:.4f}', status])

print('\n【PSI 特征监控报告】')
print(tabulate(rows, headers=['特征', 'PSI', '状态'], tablefmt='grid'))

---
## Cell 11 — Step 1 完成报告

In [ ]:
print('\n' + '='*65)
print('  📋  STEP 1 PIPELINE 完成报告（KuaiRec 版本）')
print('='*65)

summary = [
    ['数据集',         'KuaiRec（快手真实短视频）', ''],
    ['矩阵类型',       'small_matrix' if CFG['use_small_matrix'] else 'big_matrix', ''],
    ['原始交互数',     f'{len(inter_df):,}',        '行'],
    ['清洗后交互数',   f'{len(df):,}',              '行'],
    ['有效用户数',     f'{df["user_id"].nunique():,}', ''],
    ['有效视频数',     f'{df["item_id"].nunique():,}', ''],
    ['训练样本数',     f'{len(train_df):,}',         '条'],
    ['测试样本数',     f'{len(test_df):,}',          '条'],
    ['用户特征维度',   f'{uf_final.shape[1]-1}',    '列'],
    ['视频特征维度',   f'{if_final.shape[1]-1}',    '列'],
    ['Feature Store', f'{len(feature_store):,}',   'keys'],
    ['正样本阈值',     f'watch_ratio >= {CFG["watch_ratio_threshold"]}', ''],
]
print(tabulate(summary, headers=['指标', '数值', '单位'], tablefmt='simple'))

print('\n【PSI 监控汇总】')
print(tabulate(rows, headers=['特征', 'PSI', '状态'], tablefmt='simple'))

print('\n【输出文件】')
output_files = [
    f"{PATHS['cleaned']}/interactions_cleaned.parquet",
    f"{PATHS['features']}/user_features.parquet",
    f"{PATHS['features']}/item_features.parquet",
    f"{PATHS['features']}/user_sequences.parquet",
    f"{PATHS['samples']}/train.parquet",
    f"{PATHS['samples']}/test.parquet",
    f"{PATHS['feature_store']}/snapshot.pkl",
]
for fp in output_files:
    if os.path.exists(fp):
        mb = os.path.getsize(fp)/1024**2
        print(f'  ✅ {fp.replace(BASE_DIR+"/",""):50s} {mb:.2f} MB')
    else:
        print(f'  ❌ {fp}')

print('\n' + '='*65)
print('  ✅  Step 1 完成！')
print('  ▶   下一步: Step 2 — 双塔召回模型')
print('='*65)